Deals with conflicting libraries

For faster results run everything using a T4 GPU

Make sure to add the Nist file to the files section

Imports libraries and defines the theme for the table

In [13]:
%pip install qdrant-client sentence-transformers rich
%pip install rich-theme-manager

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

# --- STEP 2: VISUAL SETUP ---
from rich.style import Style
from rich_theme_manager import Theme, ThemeManager
import pathlib
import pandas as pd
import warnings
import sys

# --- STEP 4: SETUP VECTOR DB ---
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# Define Theme
THEMES = [
    Theme(
        name="dark",
        description="Dark mode theme",
        tags=["dark"],
        styles={
            "repr.own": Style(color="#e87d3e", bold=True),
            "repr.tag_name": "dim cyan",
            "repr.call": "bright_yellow",
            "repr.str": "bright_green",
            "repr.number": "bright_red",
            "repr.none": "dim white",
            "repr.attrib_name": Style(color="#e87d3e", bold=True),
            "repr.attrib_value": "bright_blue",
            "default": "bright_white on black"
        },
    )
]
theme_dir = pathlib.Path("themes").expanduser()
theme_dir.mkdir(parents=True, exist_ok=True)
theme_manager = ThemeManager(theme_dir=theme_dir, themes=THEMES)
console = Console(theme=theme_manager.get("dark"))

Initial Setup and Loads Dataset

In [15]:
# --- SETUP (Fast Reload) ---
console = Console()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen2.5-3B-Instruct"

# --- STEP 3: LOAD DATA ---
warnings.filterwarnings('ignore')
console.print("[bold green]Loading NIST Data...[/bold green]")

try:
    df = pd.read_csv('/kaggle/input/datasets/owenmcdaniel/rag-capstone/nist_controls.csv')
    df['combined_text'] = (
        "Control ID: " + df['identifier'].astype(str) + "; " +
        "Title: " + df['name'].astype(str) + "; " +
        "Control Text: " + df['control_text'].fillna('').astype(str) + "; " +
        "Discussion: " + df['discussion'].fillna('').astype(str)
    )
    data = df.to_dict('records')
    console.print(f"[dim]Loaded {len(data)} rows successfully.[/dim]")
except FileNotFoundError:
    console.print("[bold red]CRITICAL ERROR: CSV file not found![/bold red]")
    console.print("[yellow]Please re-upload 'NIST_SP-800-53.csv' to the Files panel.[/yellow]")
    data = []

# 1. Load Resources (Only if not already loaded to save time)
if 'qdrant' not in globals():
    console.print("[bold yellow]Reloading Database connection...[/bold yellow]")
    qdrant = QdrantClient(":memory:")
    encoder = SentenceTransformer('all-MiniLM-L6-v2')

    # Index NIST if data exists
    if 'data' in globals() and data:
        qdrant.recreate_collection(
            collection_name="nist_controls",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        qdrant.upload_points(
            collection_name="nist_controls",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(data)
            ]
        )
        console.print("[bold green]NIST collection indexed.[/bold green]")

    # --- LOAD AND INDEX MITRE D3FEND (separate collection) ---
    console.print("[bold green]Loading MITRE D3FEND Data...[/bold green]")
    try:
        d3fend_path = '/kaggle/input/datasets/owenmcdaniel/rag-capstone/d3fend.csv'  # adjust if needed
        df_d3fend = pd.read_csv(d3fend_path)
        
        # Create combined_text – adapt if column names differ slightly
        df_d3fend['combined_text'] = (
            "D3FEND ID: " + df_d3fend['ID'].astype(str) + "; " +
            "Tactic: " + df_d3fend.get('D3FEND Tactic', '').astype(str) + "; " +
            "Technique: " + df_d3fend.get('D3FEND Technique', '').astype(str) + "; " +
            "Level 0: " + df_d3fend.get('D3FEND Technique Level 0', '').astype(str) + "; " +
            "Level 1: " + df_d3fend.get('D3FEND Technique Level 1', '').astype(str) + "; " +
            "Definition: " + df_d3fend['Definition'].fillna('').astype(str)
        )
        
        d3fend_data = df_d3fend.to_dict('records')
        console.print(f"[dim]Loaded {len(d3fend_data)} D3FEND techniques successfully.[/dim]")
        
        # Create separate collection for D3FEND
        qdrant.recreate_collection(
            collection_name="d3fend_techniques",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        
        qdrant.upload_points(
            collection_name="d3fend_techniques",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(d3fend_data)
            ]
        )
        console.print("[bold green]D3FEND collection indexed.[/bold green]")
    
    except FileNotFoundError:
        console.print("[bold red]D3FEND CSV not found! Path checked: " + d3fend_path + "[/bold red]")
        d3fend_data = []
    # --- LOAD AND INDEX CISA KEV (Known Exploited Vulnerabilities) ---
    console.print("[bold green]Loading CISA KEV Data...[/bold green]")
    try:
        kev_path = '/kaggle/input/datasets/owenmcdaniel/rag-capstone/known_exploited_vulnerabilities.csv'  # Adjust path if needed
        df_kev = pd.read_csv(kev_path)
        
        # Construct combined_text for KEV
        df_kev['combined_text'] = (
            "CVE ID: " + df_kev['cveID'].astype(str) + "; " +
            "Vendor: " + df_kev['vendorProject'].astype(str) + "; " +
            "Product: " + df_kev['product'].astype(str) + "; " +
            "Vulnerability: " + df_kev['vulnerabilityName'].astype(str) + "; " +
            "Description: " + df_kev['shortDescription'].fillna('').astype(str) + "; " +
            "Required Action: " + df_kev['requiredAction'].fillna('').astype(str) + "; " +
            "CWEs: " + df_kev['cwes'].fillna('').astype(str)
        )
        
        kev_data = df_kev.to_dict('records')
        console.print(f"[dim]Loaded {len(kev_data)} KEV records successfully.[/dim]")
        
        # Create collection for KEV
        qdrant.recreate_collection(
            collection_name="known_exploited_vulnerabilities",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        
        qdrant.upload_points(
            collection_name="known_exploited_vulnerabilities",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(kev_data)
            ]
        )
        console.print("[bold green]CISA KEV collection indexed.[/bold green]")
        
    except FileNotFoundError:
        console.print(f"[bold red]KEV CSV not found! Path checked: {kev_path}[/bold red]")
        kev_data = []

    # --- LOAD AND INDEX MITRE ENTERPRISE ATT&CK ---
    console.print("[bold green]Loading MITRE Enterprise ATT&CK Data...[/bold green]")
    try:
        mitre_path = '/kaggle/input/datasets/owenmcdaniel/rag-capstone/mitre-enterprise-attack-v18.1.csv'  # Adjust path if needed
        df_mitre = pd.read_csv(mitre_path)
        
        # Construct combined_text for MITRE ATT&CK
        # We focus on ID, Name, Tactics, and Description for better retrieval
        df_mitre['combined_text'] = (
            "MITRE ID: " + df_mitre['ID'].astype(str) + "; " +
            "Technique Name: " + df_mitre['name'].astype(str) + "; " +
            "Tactics: " + df_mitre['tactics'].fillna('').astype(str) + "; " +
            "Platforms: " + df_mitre['platforms'].fillna('').astype(str) + "; " +
            "Description: " + df_mitre['description'].fillna('').astype(str)
        )
        
        mitre_data = df_mitre.to_dict('records')
        console.print(f"[dim]Loaded {len(mitre_data)} MITRE techniques successfully.[/dim]")
        
        # Create collection for MITRE ATT&CK
        qdrant.recreate_collection(
            collection_name="mitre_attack",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        
        qdrant.upload_points(
            collection_name="mitre_attack",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(mitre_data)
            ]
        )
        console.print("[bold green]MITRE ATT&CK collection indexed.[/bold green]")
        
    except FileNotFoundError:
        console.print(f"[bold red]MITRE CSV not found! Path checked: {mitre_path}[/bold red]")
        mitre_data = []

if 'model' not in globals():
    console.print(f"[bold yellow]Loading Model ({device})...[/bold yellow]")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)

Loading NIST Data...

Loaded 1189 rows successfully.

Chat Loop

In [ ]:
# --- MAIN CHAT LOOP ---
console.print("\n[bold green]✅ System Ready! Type 'exit' or 'quit' to stop.[/bold green]")
while True:
    # 1. Get Question
    console.print("\n[bold cyan]Your Question:[/bold cyan]")
    user_prompt = input(">>> ")
    if user_prompt.lower() in ["exit", "quit", "stop"]:
        console.print("[bold red]Stopping program. Goodbye![/bold red]")
        break
    if not user_prompt.strip():
        continue

    # 2. Search Databases
    query_vector = encoder.encode(user_prompt).tolist()
    
    # NIST search
    try:
        nist_hits = qdrant.search(collection_name="nist_controls", query_vector=query_vector, limit=5)
    except AttributeError:
        nist_hits = qdrant.query_points(collection_name="nist_controls", query=query_vector, limit=5).points
    
    # D3FEND search
    try:
        d3fend_hits = qdrant.search(collection_name="d3fend_techniques", query_vector=query_vector, limit=5)
    except AttributeError:
        d3fend_hits = qdrant.query_points(collection_name="d3fend_techniques", query=query_vector, limit=5).points

    # KEV search (New)
    try:
        kev_hits = qdrant.search(collection_name="known_exploited_vulnerabilities", query_vector=query_vector, limit=5)
    except AttributeError:
        kev_hits = qdrant.query_points(collection_name="known_exploited_vulnerabilities", query=query_vector, limit=5).points

    # ATT&CK search (New)
    try:
        attack_hits = qdrant.search(collection_name="mitre_attack", query_vector=query_vector, limit=5)
    except AttributeError:
        attack_hits = qdrant.query_points(collection_name="mitre_attack", query=query_vector, limit=5).points

    # 3. Prepare Contexts for LLM
    nist_context = "\n".join([
        f"- ID: {hit.payload.get('identifier')} | Title: {hit.payload.get('name')} | Text: {hit.payload.get('control_text')}"
        for hit in nist_hits
    ]) if nist_hits else "No relevant NIST controls found."

    d3fend_context = "\n".join([
        f"- ID: {hit.payload.get('ID')} | Technique: {hit.payload.get('D3FEND Technique', 'N/A')} | Definition: {hit.payload.get('Definition', 'N/A')}"
        for hit in d3fend_hits
    ]) if d3fend_hits else "No relevant D3FEND techniques found."

    kev_context = "\n".join([
        f"- CVE: {hit.payload.get('cveID')} | Vendor/Product: {hit.payload.get('vendorProject')} {hit.payload.get('product')} | Desc: {hit.payload.get('shortDescription')}"
        for hit in kev_hits
    ]) if kev_hits else "No relevant known vulnerabilities (KEV) found."

    attack_context = "\n".join([
        f"- Technique ID: {hit.payload.get('ID')} | Name: {hit.payload.get('name')} | Tactics: {hit.payload.get('tactics')}"
        for hit in attack_hits
    ]) if attack_hits else "No relevant MITRE ATT&CK techniques found."

    # 4. Display Tables
    from rich.table import Table
    
    # NIST Table
    nist_table = Table(title="NIST 800-53 Controls", show_lines=True)
    nist_table.add_column("Control ID", style="bright_red")
    nist_table.add_column("Title", style="green")
    nist_table.add_column("Snippet", style="yellow")
    nist_table.add_column("Score", style="#a6accd")
    for hit in nist_hits:
        snippet = str(hit.payload.get("control_text", ""))[:120] + "..."
        nist_table.add_row(str(hit.payload.get("identifier", "N/A")), str(hit.payload.get("name", "N/A")), snippet, f"{hit.score:.4f}")
    console.print(nist_table)
    
    # D3FEND Table
    d3fend_table = Table(title="MITRE D3FEND Matrix", show_lines=True)
    d3fend_table.add_column("ID", style="bright_magenta", no_wrap=True)
    d3fend_table.add_column("Tactic", style="magenta")
    d3fend_table.add_column("Technique", style="cyan")
    d3fend_table.add_column("Definition Snippet", style="yellow")
    d3fend_table.add_column("Score", style="dim")
    for hit in d3fend_hits:
        d3fend_table.add_row(
            str(hit.payload.get("ID", "N/A")),
            str(hit.payload.get("D3FEND Tactic", "N/A")),
            str(hit.payload.get("D3FEND Technique", "N/A")),
            str(hit.payload.get("Definition", ""))[:100] + "...",
            f"{hit.score:.4f}"
        )
    console.print(d3fend_table)

    # KEV Table 
    kev_table = Table(title="CISA Known Exploited Vulnerabilities", show_lines=True)
    kev_table.add_column("CVE ID", style="bright_yellow", no_wrap=True)
    kev_table.add_column("Vulnerability Name", style="white")
    kev_table.add_column("Ransomware Use", style="bold red" if "Known" else "white")
    kev_table.add_column("Required Action", style="italic")
    kev_table.add_column("Score", style="dim")
    for hit in kev_hits:
        kev_table.add_row(
            str(hit.payload.get("cveID", "N/A")),
            str(hit.payload.get("vulnerabilityName", "N/A")),
            str(hit.payload.get("knownRansomwareCampaignUse", "Unknown")),
            str(hit.payload.get("requiredAction", ""))[:80] + "...",
            f"{hit.score:.4f}"
        )
    console.print(kev_table)

    # ATT&CK Table 
    attack_table = Table(title="MITRE Enterprise ATT&CK", show_lines=True)
    attack_table.add_column("ID", style="bright_blue", no_wrap=True)
    attack_table.add_column("Technique Name", style="bright_green")
    attack_table.add_column("Tactics", style="magenta")
    attack_table.add_column("Platforms", style="white")
    attack_table.add_column("Sub?", style="italic")
    attack_table.add_column("Score", style="dim")
    for hit in attack_hits:
        attack_table.add_row(
            str(hit.payload.get("ID", "N/A")),
            str(hit.payload.get("name", "N/A")),
            str(hit.payload.get("tactics", "N/A")),
            str(hit.payload.get("platforms", "N/A"))[:40] + "...",
            "Yes" if hit.payload.get("is sub-technique") else "No",
            f"{hit.score:.4f}"
        )
    console.print(attack_table)

    # Combine contexts for LLM
    combined_context = (
        f"--- NIST CONTROLS ---\n{nist_context}\n\n"
        f"--- MITRE D3FEND ---\n{d3fend_context}\n\n"
        f"--- CISA KEV ---\n{kev_context}\n\n"
        f"--- MITRE ATT&CK ---\n{attack_context}"
    )

    # 5. Stream Answer
    messages = [
        {"role": "system", "content": """You are an expert Cybersecurity Advisor specializing in NIST SP 800-53, MITRE D3FEND, CISA KEV, and MITRE ATT&CK Enterprise. Your goal is to provide small and resource-constrained organizations with practical, high-impact security guidance.

---CORE OPERATING RULES:
1. RELEVANCE FILTER: Only include NIST controls, D3FEND techniques, and ATT&CK methods that directly solve the user's problem. If a retrieved item is physically or technically irrelevant, DISREGARD it, even if it is in the context.
2. SOURCE TRUTH: Use ONLY the provided context. If the context is insufficient, state: "I do not have enough information in the provided context to answer this."
3. NO HALLUCINATION: Do not invent control IDs or technical requirements.
4. TONE: Use plain, professional language suitable for a non-technical small business owner. Explain "the why" before "the how."

---RESPONSE STRUCTURE:
Issue Summary:
Explain the risk in plain language. Focus on the real-world consequence (e.g., "A hacker could steal your customer list if...") rather than just naming the threat.

Recommended Mitigations:
Provide specific, actionable steps a small team can take. Prioritize low-cost, high-impact actions (like MFA, backups, or training).

Relevant NIST Controls:
List only the most relevant controls. For each:
- [Control ID]: Provide a "Small Business Translation." Do not just copy the NIST snippet; explain specifically how this control protects the user's business in this scenario.

Relevant D3FEND Controls:
List only the most relevant techniques. For each:
- [D3FEND ID]: Provide a "Small Business Translation." Do not just copy the D3FEND snippet; explain specifically how this countermeasure protects the user's business in this scenario.

Relevant KEV Entries:
List only the most relevant vulnerabilities. Reference CISA KEV if the vulnerability is known to be actively exploited.
For each:
- [CVE ID]: Provide a "Real-World Risk Translation." Explain if hackers are actively using this and why it matters for their specific software or setup.

Relevant ATT&CK Techniques:
List only the most relevant adversary methods. Reference MITRE ATT&CK to explain the adversary's offensive strategy.
For each:
- [Technique ID]: Provide a "Hacker Playbook Translation." Explain in simple terms how a hacker would use this specific technique to attack a business like theirs.

---STRICT LIMITATIONS:
- Prioritize accuracy and practical utility over length. 
- Never reference external standards (ISO, CIS) unless they are in the context.
- Avoid repetitive mapping (e.g., don't list both a parent control and its enhancement unless both add unique value)."""},
        {"role": "user", "content": f"Context:\n{combined_context}\n\nQuestion: {user_prompt}\n\nAnswer:"}
    ]
    
    # Get the input IDs
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
   
    # Create the attention_mask (all 1s, same shape as input_ids)
    attention_mask = torch.ones_like(input_ids)
   
    # Initialize Streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
   
    # Pass BOTH input_ids and attention_mask to the generator
    generation_kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        streamer=streamer,
        max_new_tokens=1200
    )
   
    # Run generation in a separate thread
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    console.print("\n[bold green]Comprehensive Security Advisory:[/bold green]")
    for new_text in streamer:
        print(new_text, end="", flush=True)
    print() # Newline at end

✅ System Ready! Type 'exit' or 'quit' to stop.

Your Question:

>>>  I need to setup my network with many different departments. How do I do that safely?


                                               NIST 800-53 Controls                                                
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Control ID ┃ Title                                       ┃ Snippet                                     ┃ Score  ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ SC-22      │ Architecture and Provisioning for           │ Ensure the systems that collectively        │ 0.3948 │
│            │ Name/Address Resolution Service             │ provide name/address resolution service for │        │
│            │                                             │ an organization are fault-tolerant and ...  │        │
├────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┼────────┤
│ CA-6(1)    │ Authorization | Joint Authorization —       │ Employ a joint authorization process for    │ 0.3885 │
│            │ Intra-organization                          │ the system that includes multiple           │        │
│            │                                             │ authorizing officials from the same         │        │
│            │                                             │ organizat...                                │        │
├────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┼────────┤
│ SC-46      │ Cross Domain Policy Enforcement             │ Implement a policy enforcement mechanism    │ 0.3851 │
│            │                                             │ [Selection: physically; logically] between  │        │
│            │                                             │ the physical and/or network interfac...     │        │
├────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┼────────┤
│ SC-7(3)    │ Boundary Protection | Access Points         │ Limit the number of external network        │ 0.3816 │
│            │                                             │ connections to the system....               │        │
├────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┼────────┤
│ CP-8(3)    │ Telecommunications Services | Separation of │ Obtain alternate telecommunications         │ 0.3779 │
│            │ Primary and Alternate Providers             │ services from providers that are separated  │        │
│            │                                             │ from primary service providers to reduce    │        │
│            │                                             │ ...                                         │        │
└────────────┴─────────────────────────────────────────────┴─────────────────────────────────────────────┴────────┘

                                                MITRE D3FEND Matrix                                                
┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ ID      ┃ Tactic  ┃ Technique ┃ Definition Snippet                                                     ┃ Score  ┃
┡━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ D3-BDI  │ Isolate │ nan       │ Broadcast isolation restricts the number of computers a host can       │ 0.2918 │
│         │         │           │ contact on their LAN....                                               │        │
├─────────┼─────────┼───────────┼────────────────────────────────────────────────────────────────────────┼────────┤
│ D3-ANAA │ Detect  │ nan       │ Detection of unauthorized use of administrative network protocols by   │ 0.2783 │
│         │         │           │ analyzing network activity agai...                                     │        │
├─────────┼─────────┼───────────┼────────────────────────────────────────────────────────────────────────┼────────┤
│ D3-NTCD │ Detect  │ nan       │ Establishing baseline communities of network hosts and identifying     │ 0.2672 │
│         │         │           │ statistically divergent inter-com...                                   │        │
├─────────┼─────────┼───────────┼────────────────────────────────────────────────────────────────────────┼────────┤
│ D3-NVA  │ Model   │ nan       │ Network vulnerability assessment relates all the vulnerabilities of a  │ 0.2579 │
│         │         │           │ network's components in the co...                                      │        │
├─────────┼─────────┼───────────┼────────────────────────────────────────────────────────────────────────┼────────┤
│ D3-RPA  │ Detect  │ nan       │ The detection of an internal host relaying traffic between the         │ 0.2474 │
│         │         │           │ internal network and the external net...                               │        │
└─────────┴─────────┴───────────┴────────────────────────────────────────────────────────────────────────┴────────┘

                                       CISA Known Exploited Vulnerabilities                                        
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ CVE ID         ┃ Vulnerability Name                ┃ Ransomware Use ┃ Required Action                  ┃ Score  ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ CVE-2015-1187  │ D-Link and TRENDnet Multiple      │ Unknown        │ The impacted product is          │ 0.2517 │
│                │ Devices Remote Code Execution     │                │ end-of-life and should be        │        │
│                │ Vulnerability                     │                │ disconnected if still in use.... │        │
├────────────────┼───────────────────────────────────┼────────────────┼──────────────────────────────────┼────────┤
│ CVE-2019-16920 │ D-Link Multiple Routers Command   │ Unknown        │ The impacted product is          │ 0.2293 │
│                │ Injection Vulnerability           │                │ end-of-life and should be        │        │
│                │                                   │                │ disconnected if still in use.... │        │
├────────────────┼───────────────────────────────────┼────────────────┼──────────────────────────────────┼────────┤
│ CVE-2015-2051  │ D-Link DIR-645 Router Remote Code │ Unknown        │ The impacted product is          │ 0.1755 │
│                │ Execution Vulnerability           │                │ end-of-life and should be        │        │
│                │                                   │                │ disconnected if still in use.... │        │
├────────────────┼───────────────────────────────────┼────────────────┼──────────────────────────────────┼────────┤
│ CVE-2020-8515  │ Multiple DrayTek Vigor Routers    │ Unknown        │ Apply updates per vendor         │ 0.1742 │
│                │ Web Management Page Vulnerability │                │ instructions....                 │        │
├────────────────┼───────────────────────────────────┼────────────────┼──────────────────────────────────┼────────┤
│ CVE-2024-9474  │ Palo Alto Networks PAN-OS         │ Known          │ Apply mitigations per vendor     │ 0.1657 │
│                │ Management Interface OS Command   │                │ instructions or discontinue use  │        │
│                │ Injection Vulnerability           │                │ of the product if m...           │        │
└────────────────┴───────────────────────────────────┴────────────────┴──────────────────────────────────┴────────┘

                                              MITRE Enterprise ATT&CK                                              
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┓
┃ ID        ┃ Technique Name             ┃ Tactics                   ┃ Platforms                  ┃ Sub? ┃ Score  ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━┩
│ T1599     │ Network Boundary Bridging  │ Defense Evasion           │ Network Devices...         │ No   │ 0.3246 │
├───────────┼────────────────────────────┼───────────────────────────┼────────────────────────────┼──────┼────────┤
│ T1104     │ Multi-Stage Channels       │ Command and Control       │ ESXi, Linux, Windows,      │ No   │ 0.2850 │
│           │                            │                           │ macOS...                   │      │        │
├───────────┼────────────────────────────┼───────────────────────────┼────────────────────────────┼──────┼────────┤
│ T1199     │ Trusted Relationship       │ Initial Access            │ IaaS, Identity Provider,   │ No   │ 0.2843 │
│           │                            │                           │ Linux, Office S...         │      │        │
├───────────┼────────────────────────────┼───────────────────────────┼────────────────────────────┼──────┼────────┤
│ T1037.003 │ Boot or Logon              │ Persistence, Privilege    │ Windows...                 │ Yes  │ 0.2771 │
│           │ Initialization Scripts:    │ Escalation                │                            │      │        │
│           │ Network Logon Script       │                           │                            │      │        │
├───────────┼────────────────────────────┼───────────────────────────┼────────────────────────────┼──────┼────────┤
│ T1602.002 │ Data from Configuration    │ Collection                │ Network Devices...         │ Yes  │ 0.2729 │
│           │ Repository: Network Device │                           │                            │      │        │
│           │ Configuration Dump         │                           │                            │      │        │
└───────────┴────────────────────────────┴───────────────────────────┴────────────────────────────┴──────┴────────┘

Comprehensive Security Advisory:

### Issue Summary
Your network consists of multiple departments, each potentially handling sensitive data. If a breach occurs, a hacker could access critical information, leading to significant financial loss and reputational damage. Ensuring that your network is secure across these departments is crucial to protect your business.

### Recommended Mitigations
1. **Implement Role-Based Access Control (RBAC)**: Assign permissions based on roles within each department. This ensures that users only have access to the resources they need to perform their job functions.
2. **Use Strong Authentication**: Require multi-factor authentication (MFA) for accessing critical systems. This adds another layer of security beyond just passwords.
3. **Limit External Connections**: Restrict the number of external network connections to your internal systems. This reduces the risk of unauthorized access.
4. **Regular Security Audits**: Conduct regular security audits to ensure compliance with your security

Your Question: